In [121]:
# # Delete all R objects and run garbage collection so we start with a clean slate
# rm(list = ls())
# invisible(gc())

# thread_offset <- 0

# sample_size_divisor <- 125

# # Whether to sample each split_part by sample_size_divisor
# # (useful when iterating through code runs in quick succession)
# to_sample <- TRUE
# # TODO: Add description here
# to_write <- TRUE
# # TODO: Add description here
# to_flush <- FALSE
# # TODO: Add description here
# to_parallel <- TRUE
# cat("Parallelization:", to_parallel, "\n")
# # TODO: Add description here
# to_debug <- FALSE
# verbose_output <- if (to_debug) TRUE else FALSE

# to_generate_subset <- TRUE

# to_py_prompt <- TRUE
# to_python <- TRUE
# to_generate_py_fwrite <- TRUE
# to_generate_feather <- TRUE
# to_py_bq <- FALSE

# to_thai_prompt <- TRUE
# to_thai <- TRUE
# to_thai_bq <- FALSE
# to_generate_thai_txt <- TRUE
# to_thai_all_years <- FALSE

# to_spc <- FALSE

source("~/drg-pipeline/data-cleaning/00a-parameters.r")


Parallelization: TRUE 


In [122]:
# Update the grouper
system("git submodule update --init --recursive")

# List, install (if applicable), and load packages
## Required packages
required_packages <- c(
  "data.table", "here", "tictoc", "stringr", "stringi", "lubridate",
  "profvis", "hash", "future", "future.apply", "knitr", "htmlwidgets",
  "parallelly", "stringdist", "parallel", "reticulate", "bigrquery",
  "jsonlite", "googleCloudStorageR", "haven", "fst", "httr", "ggplot2",
  "rmarkdown", "digest", "base64enc", "arrow"
  # , "docstring", "progress" # Comma is here so if I uncomment this line it
  # automatically works without having to type or delete a comma after haven
)

# Additional packages to install via remotes (GitHub), if not available
github_packages <- c("r-lib/styler")

# Function to install and load packages quietly
install_and_load <- function(package) {
  if (!require(package, character.only = TRUE)) {
    message("Installing ", package)
    install.packages(package, dependencies = TRUE)
  } else {
    if (verbose_output) message("Loading ", package)
  }
  library(package, character.only = TRUE)
}

# Function to install packages from GitHub via remotes
install_from_github <- function(repo) {
  package_name <- basename(repo)
  if (!require(package_name, character.only = TRUE)) {
    if (!require("remotes", character.only = TRUE)) {
      install.packages("remotes")
    }
    message("Installing ", package_name, " from GitHub (", repo, ")")
    remotes::install_github(repo)
  } else {
    if (verbose_output) message("Loading ", package_name)
  }
  library(package_name, character.only = TRUE)
}

# Apply the function to each required package
message("Installing/loading required CRAN packages...")
invisible(
  suppressPackageStartupMessages(
    lapply(required_packages, install_and_load)
  )
)

# Install and load GitHub packages if not installed
message("Installing/loading required GitHub packages...")
invisible(
  suppressPackageStartupMessages(
    lapply(github_packages, install_from_github)
  )
)

# detect available threads
nthreads <- parallelly::availableCores()


Installing/loading required CRAN packages...

Installing/loading required GitHub packages...



In [123]:
scripts_path <- here("data-cleaning/r_scripts_v2")

# List all R files in the directory with full paths, sorted by filename
r_files <- list.files(scripts_path, pattern = "\\.R$", full.names = TRUE)

# Source each file sequentially
for (file in r_files) {
  if (verbose_output) message(Sys.time(), " Sourcing: ", file)
  invisible(source(file))
}

message(year_to_load)

bq_dataset <- "drg_claims"


All directories exist.


Total Rows via cached object: 12562622

Utilizing 12 cores (24 threads)


2019



In [124]:
dt <- fread(here(checkpoint_9_path, paste0("checkpoint_9_grouper_differences_", year_to_load, suffix, ".csv")), colClasses = "character")


In [125]:
claims <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_bdate_with_time", ".rds")))


In [126]:
result <- merge(dt, claims, by = "id_series", all.x = TRUE)


In [127]:
fwrite(result, paste0("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_9_grouper_differences/checkpoint_9_grouper_differences_with_full_claims_data_", year_to_load, suffix, ".csv"))


In [128]:
result <- fread(paste0("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_9_grouper_differences/checkpoint_9_grouper_differences_with_full_claims_data_", year_to_load, suffix, ".csv"), colClasses = "character")


In [129]:
thai_input <- fread(paste0("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_4_thai_master_input/checkpoint_4_thai_grouper_input_", year_to_load, suffix, "part_1_of_1.txt"), colClasses = "character")


In [130]:
thai_result <- fread(paste0("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_5_thai_output/PRE-TDRG_CHECKPOINT_4_THAI_GROUPER_INPUT_", toupper(paste0(year_to_load, suffix)), "PART_1_OF_1Res.TXT"), colClasses = "character")


In [131]:
py_input <- as.data.table(read_feather(paste0("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_7_py_input/python_final_input_", year_to_load, suffix, ".feather")))


In [132]:
py_output <- as.data.table(read_feather(paste0("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_8_py_output/python_output_", year_to_load, suffix, ".feather")))


In [133]:
# str(result)


In [134]:
# str(thai_input)


In [135]:
# str(thai_result)


In [136]:
# str(py_input)


In [137]:
# str(py_output)


In [138]:
# Assuming result, thai_input, and thai_result are already data.tables
# Rename columns to have a common key column for merging
setnames(result, "row", "key")
setnames(thai_input, "CASEID", "key")
setnames(thai_result, "caseid", "key")

# Perform the three-way merge
merged_data <- merge(result, thai_input, by = "key", all = FALSE) # Merge result and thai_input
merged_data <- merge(merged_data, thai_result, by = "key", all = FALSE) # Merge with thai_result

# View the resulting data.table
# print(merged_data)


In [139]:
# Rename the columns in py_input and py_output to have a common name for merging
setnames(py_input, "id_series", "key_series")
setnames(py_output, "id_series", "key_series")

# Rename the column in the merged data for consistency
setnames(merged_data, "id_series", "key_series")

# Merge the `merged_data` with `py_input`
merged_with_py_input <- merge(merged_data, py_input, by = "key_series", all = FALSE)

# Merge the result with `py_output`
final_merged_data <- merge(merged_with_py_input, py_output, by = "key_series", all = FALSE)

# View the final merged data
# print(final_merged_data)


In [140]:
# Construct the file path dynamically
file_path <- paste0(
  "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_9_grouper_differences/final_merged_data_",
  year_to_load,
  suffix,
  ".csv"
)

# Save the `final_merged_data` data.table as a CSV file
fwrite(final_merged_data, file_path)

# Confirm the file has been saved
# cat("File saved to:", file_path, "\n")
